Датасет SUES-200 из https://github.com/Reza-Zhu/SUES-200-Benchmark

In [1]:
!pip install -U --no-cache-dir gdown timm tqdm

import os
import zipfile

from google.colab import drive

drive.mount('/content/drive')

zip_path_on_drive = '/content/drive/MyDrive/SUES-200-512x512-V2.zip'
extract_dir = '/content/sues200_dataset'

if os.path.exists(zip_path_on_drive):
    if not os.path.exists(extract_dir):
        with zipfile.ZipFile(zip_path_on_drive, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Датасет распакован в: {extract_dir}")

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.1
    Uninstalling gdown-5.2.1:
      Successfully uninstalled gdown-5.2.1
Mounted at /content/drive
Датасет распакован в: /content/sues200_dataset


обработка снимков с дрона и спутника (изменение отдельно для спутниковых, сжатие, полярные и не полярные координаты)

In [2]:
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm


#обработка для спутниковых снимков (яркость и контраст)
def clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe_obj.apply(l)
    limg = cv2.merge((cl,a,b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def process_and_save(src_dir, dest_dir, mode='cartesian'):
    src_path = Path(src_dir)
    dest_path = Path(dest_dir)
    all_images = list(src_path.rglob('*.jpg')) + list(src_path.rglob('*.png'))

    for img_path in tqdm(all_images, desc=f"Processing {mode}"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if 'satellite-view' in str(img_path):
            img = clahe(img)

        if mode == 'cartesian':
            img_processed = cv2.resize(img, (224, 224))
        elif mode == 'polar':
            img_resized = cv2.resize(img, (256, 256))
            center = (128, 128)
            max_radius = 128
            img_polar = cv2.warpPolar(img_resized, (256, 256), center, max_radius, cv2.WARP_POLAR_LINEAR)
            start = (256 - 224) // 2
            img_processed = img_polar[start:start+224, start:start+224]

        rel_path = img_path.relative_to(src_path)
        save_path = dest_path / rel_path

        save_path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img_processed).save(save_path)

ORIGINAL_DATASET_DIR = "/content/sues200_dataset/SUES-200-512x512"
CARTESIAN_OUT = "/content/sues_cartesian"
POLAR_OUT = "/content/sues_polar"

if os.path.exists(ORIGINAL_DATASET_DIR):
    if not os.path.exists(CARTESIAN_OUT):
        process_and_save(ORIGINAL_DATASET_DIR, CARTESIAN_OUT, mode='cartesian')
    if not os.path.exists(POLAR_OUT):
        process_and_save(ORIGINAL_DATASET_DIR, POLAR_OUT, mode='polar')

Processing polar: 100%|██████████| 40200/40200 [04:14<00:00, 157.92it/s]


добавление искажений (изменение яркости, сдвиг для полярных, повторо до 180 градусов для не полярных), разделение на обучающую и тестовую

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


#сдвиг для полярных координат
class HorizontalRoll:
    def __call__(self, img):
        shift = torch.randint(0, img.shape[2], (1,)).item()
        return torch.roll(img, shifts=shift, dims=2)

def get_transforms(mode='cartesian', is_train=True):
    if not is_train:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    base_t = [transforms.ColorJitter(brightness=0.2, contrast=0.2)]
    if mode == 'cartesian':
        base_t.insert(0, transforms.RandomRotation(180))
        base_t.append(transforms.ToTensor())
    elif mode == 'polar':
        base_t.append(transforms.ToTensor())
        base_t.append(HorizontalRoll())

    base_t.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    return transforms.Compose(base_t)

class SUESDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train', view='all'):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.images, self.labels, self.is_drone, self.heights = [], [], [], []
        loc_range = range(1, 121) if split == 'train' else range(121, 201)

        views = []
        if view in ['all', 'drone']: views.append('drone_view_512')
        if view in ['all', 'satellite']: views.append('satellite-view')

        for v_type in views:
            v_path = self.root_dir / v_type
            for img_path in v_path.rglob('*'):
                if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                    loc_num = None
                    for part in img_path.parts:
                        if part.isdigit() and len(part) == 4:
                            loc_num = int(part)
                            break
                    if loc_num is None or loc_num not in loc_range:
                        continue

                    self.images.append(str(img_path))
                    self.labels.append(loc_num)
                    drn = 'drone' in v_type
                    self.is_drone.append(drn)

                    h = 0
                    if drn:
                        for val in ['150', '200', '250', '300']:
                            if f"/{val}/" in str(img_path) or val in img_path.name:
                                h = int(val)
                                break
                    self.heights.append(h)
        print(f"Загружен {split} ({view}): {len(self.images)} фото")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx], self.is_drone[idx], self.heights[idx]

https://github.com/Gongnaiqun7/MIFT/blob/main/models/MIFT/make_model.py

первые два слоя раздельные, следующие - общие

In [4]:
import timm


class DualBranchSwin(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch_drone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0, global_pool='avg')
        self.branch_sat = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0, global_pool='avg')

        #общие слои
        for i in range(2, len(self.branch_drone.layers)):
            self.branch_drone.layers[i] = self.branch_sat.layers[i]

        self.branch_drone.norm = self.branch_sat.norm
        self.num_features = self.branch_drone.num_features

    def forward(self, x, is_drone):
        feat = torch.zeros(x.shape[0], self.num_features, device=x.device)
        drone_mask = is_drone.bool()
        sat_mask = ~drone_mask

        if drone_mask.any():
            feat[drone_mask] = self.branch_drone(x[drone_mask])
        if sat_mask.any():
            feat[sat_mask] = self.branch_sat(x[sat_mask])

        return F.normalize(feat, p=2, dim=1)

class CircleLoss(nn.Module):
    def __init__(self, m=0.25, gamma=256):
        super().__init__()
        self.m, self.gamma = m, gamma

    def forward(self, feat, labels):
        sim = torch.matmul(feat, feat.T)
        lbl = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_m = lbl & ~torch.eye(labels.size(0), dtype=torch.bool, device=feat.device)
        neg_m = ~lbl

        sp, sn = sim[pos_m], sim[neg_m]
        if len(sp) == 0 or len(sn) == 0:
            return (feat * 0.0).sum()

        ap = torch.relu(-sp.detach() + 1 + self.m)
        an = torch.relu(sn.detach() + self.m)

        lp = -self.gamma * ap * (sp - (1 - self.m))
        ln = self.gamma * an * (sn - self.m)

        return torch.logsumexp(ln, dim=0) + torch.logsumexp(lp, dim=0)

оценка качества

In [8]:
def extract_features(model, loader, is_drone_flag, device):
    feats, labels, heights = [], [], []
    with torch.no_grad():
        for img, lbl, _, h in tqdm(loader, desc="Извлечение признаков"):
            img = img.to(device)
            dr_flag = torch.full((img.size(0),), is_drone_flag, device=device)
            with torch.amp.autocast('cuda'):
                feat = model(img, dr_flag)
            feats.append(feat.cpu())
            labels.append(lbl)
            heights.append(h)

    if len(feats) == 0:
        return torch.tensor([]), torch.tensor([]), torch.tensor([])
    return torch.cat(feats), torch.cat(labels), torch.cat(heights)

def calculate_metrics(query_feat, query_labels, gallery_feat, gallery_labels):
    dist_matrix = torch.matmul(query_feat, gallery_feat.T)
    indices = torch.argsort(dist_matrix, dim=1, descending=True)

    recall_1, recall_5, recall_10 = 0, 0, 0
    all_ap = []

    for i in range(len(query_labels)):
        label = query_labels[i]
        ranked_labels = gallery_labels[indices[i]]
        correct_pos = (ranked_labels == label).nonzero(as_tuple=False).flatten()

        if len(correct_pos) > 0:
            first_match = correct_pos[0].item()
            if first_match < 1: recall_1 += 1
            if first_match < 5: recall_5 += 1
            if first_match < 10: recall_10 += 1

            ap = 0.0
            for j, pos in enumerate(correct_pos):
                ap += (j + 1) / (pos.item() + 1)
            ap /= len(correct_pos)
            all_ap.append(ap)
        else:
            all_ap.append(0.0)

    num_queries = len(query_labels)
    return {
        "R@1": recall_1 / num_queries,
        "R@5": recall_5 / num_queries,
        "R@10": recall_10 / num_queries,
        "mAP": np.mean(all_ap)
    }

def evaluate_model(model, mode, root_data):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    query_dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=False), split='test', view='drone')
    gallery_dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=False), split='test', view='satellite')

    query_loader = DataLoader(query_dataset, batch_size=32, shuffle=False, num_workers=2)
    gallery_loader = DataLoader(gallery_dataset, batch_size=32, shuffle=False, num_workers=2)

    gallery_feat, gallery_labels, _ = extract_features(model, gallery_loader, False, device)
    query_feat, query_labels, query_heights = extract_features(model, query_loader, True, device)

    if query_feat.numel() == 0 or gallery_feat.numel() == 0:
        return None, 0.0

    results = calculate_metrics(query_feat, query_labels, gallery_feat, gallery_labels)

    print("\n")
    print(f"mAP: {results['mAP']:.4f} | R@1: {results['R@1']:.4f} | R@5: {results['R@5']:.4f} | R@10: {results['R@10']:.4f}")

    unique_heights = [150, 200, 250, 300]
    for h_val in unique_heights:
        mask = (query_heights == h_val)
        if mask.any():
            h_feat = query_feat[mask]
            h_labels = query_labels[mask]
            h_res = calculate_metrics(h_feat, h_labels, gallery_feat, gallery_labels)
            print(f"Высота {h_val}м: mAP: {h_res['mAP']:.4f} | R@1: {h_res['R@1']:.4f} | R@5: {h_res['R@5']:.4f} | R@10: {h_res['R@10']:.4f}")

    return results, results['mAP']

обучения (с сохранением лучшей модели)

In [6]:
def train_model(mode, root_data, epochs=5, batch_size=32):
    dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=True), split='train', view='all')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DualBranchSwin().to(device)

    checkpoint_path = f'/content/best_model_{mode}.pth'
    learning_rate = 1e-5
    best_map = 0.0
    criterion = CircleLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        loop = tqdm(loader, desc=f"Train Epoch {epoch+1}/{epochs} ({mode})")

        for img, labels, is_drone, _ in loop:
            img, labels = img.to(device), labels.to(device)
            is_drone_bool = is_drone.to(device).bool()

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                feat = model(img, is_drone_bool)
                loss = criterion(feat, labels)

            if loss.item() == 0.0:
                continue

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loop.set_postfix(loss=loss.item())

        scheduler.step()
        eval_results, current_map = evaluate_model(model, mode, root_data)

        if current_map > best_map:
            best_map = current_map
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'mAP': best_map,
            }, checkpoint_path)
            print(f"Новый лучший результат mAP для {mode.upper()}: {best_map:.4f}")

    return model

In [9]:
CARTESIAN_DIR = '/content/sues_cartesian'
if os.path.exists(CARTESIAN_DIR):
    model_cart = train_model(mode='cartesian', root_data=CARTESIAN_DIR, epochs=10)

Загружен train (all): 24120 фото


Train Epoch 1/10 (cartesian): 100%|██████████| 753/753 [03:57<00:00,  3.17it/s, loss=87.9]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:45<00:00, 10.94it/s]




mAP: 0.7323 | R@1: 0.6320 | R@5: 0.8578 | R@10: 0.9210
Высота 150м: mAP: 0.6602 | R@1: 0.5445 | R@5: 0.7983 | R@10: 0.8760
Высота 200м: mAP: 0.7329 | R@1: 0.6338 | R@5: 0.8530 | R@10: 0.9173
Высота 250м: mAP: 0.7644 | R@1: 0.6697 | R@5: 0.8862 | R@10: 0.9443
Высота 300м: mAP: 0.7717 | R@1: 0.6800 | R@5: 0.8935 | R@10: 0.9465
Новый лучший результат mAP для CARTESIAN: 0.7323


...



mAP: 0.8003 | R@1: 0.7200 | R@5: 0.9050 | R@10: 0.9485
Высота 150м: mAP: 0.7605 | R@1: 0.6705 | R@5: 0.8752 | R@10: 0.9270
Высота 200м: mAP: 0.7989 | R@1: 0.7195 | R@5: 0.9008 | R@10: 0.9440
Высота 250м: mAP: 0.8182 | R@1: 0.7398 | R@5: 0.9197 | R@10: 0.9583
Высота 300м: mAP: 0.8236 | R@1: 0.7502 | R@5: 0.9243 | R@10: 0.9647


Train Epoch 10/10 (cartesian): 100%|██████████| 753/753 [03:58<00:00,  3.16it/s, loss=52.6]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:45<00:00, 10.88it/s]




mAP: 0.8015 | R@1: 0.7216 | R@5: 0.9051 | R@10: 0.9479
Высота 150м: mAP: 0.7613 | R@1: 0.6713 | R@5: 0.8762 | R@10: 0.9270
Высота 200м: mAP: 0.7999 | R@1: 0.7208 | R@5: 0.9018 | R@10: 0.9435
Высота 250м: mAP: 0.8203 | R@1: 0.7428 | R@5: 0.9197 | R@10: 0.9580
Высота 300м: mAP: 0.8247 | R@1: 0.7515 | R@5: 0.9227 | R@10: 0.9630


In [10]:
POLAR_DIR = '/content/sues_polar'
if os.path.exists(POLAR_DIR):
    model_polar = train_model(mode='polar', root_data=POLAR_DIR, epochs=10)

Загружен train (all): 24120 фото


Train Epoch 1/10 (polar): 100%|██████████| 753/753 [04:00<00:00,  3.13it/s, loss=102]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:45<00:00, 10.87it/s]




mAP: 0.4442 | R@1: 0.3232 | R@5: 0.5709 | R@10: 0.6976
Высота 150м: mAP: 0.3735 | R@1: 0.2565 | R@5: 0.4815 | R@10: 0.6148
Высота 200м: mAP: 0.4363 | R@1: 0.3170 | R@5: 0.5560 | R@10: 0.6965
Высота 250м: mAP: 0.4672 | R@1: 0.3427 | R@5: 0.6102 | R@10: 0.7270
Высота 300м: mAP: 0.4995 | R@1: 0.3767 | R@5: 0.6360 | R@10: 0.7522
Новый лучший результат mAP для POLAR: 0.4442


...



mAP: 0.5856 | R@1: 0.4680 | R@5: 0.7259 | R@10: 0.8096
Высота 150м: mAP: 0.5136 | R@1: 0.3870 | R@5: 0.6637 | R@10: 0.7755
Высота 200м: mAP: 0.5672 | R@1: 0.4447 | R@5: 0.7188 | R@10: 0.8100
Высота 250м: mAP: 0.6195 | R@1: 0.5085 | R@5: 0.7528 | R@10: 0.8270
Высота 300м: mAP: 0.6422 | R@1: 0.5317 | R@5: 0.7682 | R@10: 0.8260


Train Epoch 10/10 (polar): 100%|██████████| 753/753 [03:51<00:00,  3.25it/s, loss=65.2]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:45<00:00, 10.95it/s]




mAP: 0.5885 | R@1: 0.4718 | R@5: 0.7284 | R@10: 0.8129
Высота 150м: mAP: 0.5146 | R@1: 0.3887 | R@5: 0.6640 | R@10: 0.7775
Высота 200м: mAP: 0.5695 | R@1: 0.4457 | R@5: 0.7215 | R@10: 0.8125
Высота 250м: mAP: 0.6228 | R@1: 0.5125 | R@5: 0.7580 | R@10: 0.8325
Высота 300м: mAP: 0.6471 | R@1: 0.5403 | R@5: 0.7700 | R@10: 0.8293


дообучение

In [11]:
def resume_training(mode, root_data, epochs=5, batch_size=32):
    dataset = SUESDataset(root_dir=root_data, transform=get_transforms(mode, is_train=True), split='train', view='all')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DualBranchSwin().to(device)

    checkpoint_path = f'/content/best_model_{mode}.pth'
    new_checkpoint_path = f'/content/best_model_{mode}_finetuned.pth'
    best_map = 0.0
    start_epoch = 0

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        best_map = checkpoint.get('mAP', 0.0)
        start_epoch = checkpoint.get('epoch', 0)
        print(f"Загружена модель с mAP: {best_map:.4f}")

    learning_rate = 0.000005
    criterion = CircleLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        model.train()
        loop = tqdm(loader, desc=f"Fine-tune Epoch {epoch+1}/{epochs} ({mode})")

        for img, labels, is_drone, _ in loop:
            img, labels = img.to(device), labels.to(device)
            is_drone_bool = is_drone.to(device).bool()

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                feat = model(img, is_drone_bool)
                loss = criterion(feat, labels)

            if loss.item() == 0.0:
                continue

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loop.set_postfix(loss=loss.item())

        scheduler.step()
        eval_results, current_map = evaluate_model(model, mode, root_data)

        if current_map > best_map:
            best_map = current_map
            torch.save({
                'epoch': start_epoch + epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'mAP': best_map,
            }, new_checkpoint_path)
            print(f"Новый лучший результат mAP для {mode.upper()}: {best_map:.4f}")

    return model

In [12]:
if os.path.exists(CARTESIAN_DIR):
    model_cart_finetuned = resume_training(mode='cartesian', root_data=CARTESIAN_DIR, epochs=5)

Загружен train (all): 24120 фото
Загружена модель с mAP: 0.8140


Fine-tune Epoch 1/5 (cartesian): 100%|██████████| 753/753 [03:58<00:00,  3.16it/s, loss=33.1]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:46<00:00, 10.73it/s]




mAP: 0.8199 | R@1: 0.7459 | R@5: 0.9127 | R@10: 0.9501
Высота 150м: mAP: 0.7708 | R@1: 0.6797 | R@5: 0.8845 | R@10: 0.9290
Высота 200м: mAP: 0.8224 | R@1: 0.7508 | R@5: 0.9137 | R@10: 0.9503
Высота 250м: mAP: 0.8408 | R@1: 0.7725 | R@5: 0.9260 | R@10: 0.9603
Высота 300м: mAP: 0.8454 | R@1: 0.7808 | R@5: 0.9267 | R@10: 0.9610
Новый лучший результат mAP для CARTESIAN: 0.8199


...



mAP: 0.8122 | R@1: 0.7388 | R@5: 0.9056 | R@10: 0.9482
Высота 150м: mAP: 0.7741 | R@1: 0.6920 | R@5: 0.8768 | R@10: 0.9287
Высота 200м: mAP: 0.8114 | R@1: 0.7370 | R@5: 0.9010 | R@10: 0.9475
Высота 250м: mAP: 0.8276 | R@1: 0.7548 | R@5: 0.9223 | R@10: 0.9583
Высота 300м: mAP: 0.8357 | R@1: 0.7712 | R@5: 0.9223 | R@10: 0.9583


Fine-tune Epoch 5/5 (cartesian): 100%|██████████| 753/753 [03:59<00:00,  3.14it/s, loss=90.1]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:46<00:00, 10.86it/s]




mAP: 0.8127 | R@1: 0.7388 | R@5: 0.9073 | R@10: 0.9485
Высота 150м: mAP: 0.7739 | R@1: 0.6903 | R@5: 0.8782 | R@10: 0.9290
Высота 200м: mAP: 0.8122 | R@1: 0.7380 | R@5: 0.9032 | R@10: 0.9473
Высота 250м: mAP: 0.8295 | R@1: 0.7568 | R@5: 0.9243 | R@10: 0.9587
Высота 300м: mAP: 0.8351 | R@1: 0.7700 | R@5: 0.9233 | R@10: 0.9590


In [13]:
if os.path.exists(POLAR_DIR):
    model_polar_finetuned = resume_training(mode='polar', root_data=POLAR_DIR, epochs=5)

Загружен train (all): 24120 фото
Загружена модель с mAP: 0.5919


Fine-tune Epoch 1/5 (polar): 100%|██████████| 753/753 [03:50<00:00,  3.26it/s, loss=67.6]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:46<00:00, 10.79it/s]




mAP: 0.5691 | R@1: 0.4494 | R@5: 0.7080 | R@10: 0.7988
Высота 150м: mAP: 0.4921 | R@1: 0.3605 | R@5: 0.6470 | R@10: 0.7582
Высота 200м: mAP: 0.5544 | R@1: 0.4315 | R@5: 0.6957 | R@10: 0.7957
Высота 250м: mAP: 0.6042 | R@1: 0.4913 | R@5: 0.7360 | R@10: 0.8207
Высота 300м: mAP: 0.6256 | R@1: 0.5142 | R@5: 0.7532 | R@10: 0.8205


...



mAP: 0.5951 | R@1: 0.4811 | R@5: 0.7272 | R@10: 0.8184
Высота 150м: mAP: 0.5245 | R@1: 0.3997 | R@5: 0.6655 | R@10: 0.7808
Высота 200м: mAP: 0.5766 | R@1: 0.4567 | R@5: 0.7185 | R@10: 0.8165
Высота 250м: mAP: 0.6258 | R@1: 0.5160 | R@5: 0.7562 | R@10: 0.8405
Высота 300м: mAP: 0.6534 | R@1: 0.5520 | R@5: 0.7685 | R@10: 0.8360


Fine-tune Epoch 5/5 (polar): 100%|██████████| 753/753 [03:52<00:00,  3.24it/s, loss=38.5]


Загружен test (drone): 16000 фото
Загружен test (satellite): 80 фото


Извлечение признаков: 100%|██████████| 500/500 [00:45<00:00, 11.00it/s]




mAP: 0.5955 | R@1: 0.4819 | R@5: 0.7270 | R@10: 0.8192
Высота 150м: mAP: 0.5260 | R@1: 0.4037 | R@5: 0.6647 | R@10: 0.7810
Высота 200м: mAP: 0.5747 | R@1: 0.4530 | R@5: 0.7167 | R@10: 0.8173
Высота 250м: mAP: 0.6263 | R@1: 0.5165 | R@5: 0.7558 | R@10: 0.8395
Высота 300м: mAP: 0.6549 | R@1: 0.5543 | R@5: 0.7708 | R@10: 0.8390
